<a href="https://colab.research.google.com/github/Sina-Haz/Tensegrity-Project/blob/main/GNN_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -Uq jraph
! pip install -Uq equinox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.4 MB/s eta 0:00:00


In [ ]:
# Copy/Paste code so we can read in our graph
import jax
import jax.numpy as np
import jraph
from flax.serialization import to_bytes, from_bytes

def save_graph_tuple(graph: jraph.GraphsTuple, filename: str) -> None:
    """Save a jraph.GraphsTuple to a binary file."""
    with open(filename, "wb") as f:
        f.write(to_bytes(graph))
    print(f"Graph saved to {filename}")

def load_graph_tuple(filename: str) -> jraph.GraphsTuple:
    """Load a jraph.GraphsTuple from a binary file."""
    with open(filename, "rb") as f:
        graph = from_bytes(jraph.GraphsTuple, f.read())
    print(f"Graph loaded from {filename}")
    graph = jraph.GraphsTuple(**graph)
    return graph

In [ ]:
graph = load_graph_tuple('tst_graph.bin')

Graph loaded from tst_graph.bin


In [ ]:
graph.nodes.keys()

dict_keys(['body_frame', 'com_dir', 'com_dist', 'height', 'inv_I', 'inv_M', 'pos', 'vel', 'local_offsets', 'local_dists'])

In [ ]:
# Try the simplest possible thing, concatenate all features together and make a single node rep and single edge rep
import equinox as eqx

@jraph.concatenated_args
def update_nodes(features):
  print(features.shape)
  return features

# Can't just concat args here b/c different edge types have different shapes in the 1st dim
# So when we don't use concatenated args our input is the edge/node dict
# @jraph.concatenated_args
def update_edges(features):
  print(type(features))
  print(features.keys())
  return features

encoder = jraph.GraphMapFeatures(
    embed_edge_fn = update_edges,
    embed_node_fn = update_nodes,
)

encoder(graph)

sum(n.shape[-1] for n in graph.nodes.values()) # so we can use this for getting node input shape

(22, 22)
<class 'dict'>
dict_keys(['body_curr_disp', 'body_curr_dist', 'body_rest_disp', 'body_rest_dist', 'cable_disp', 'cable_dist', 'cable_dir', 'cable_relV', 'cable_relV_norm', 'cable_stiffness', 'cable_damping', 'cable_stiffness_mag', 'cable_damping_mag', 'cable_rest_len', 'con_dist', 'con_normal_dir', 'con_normal_relV', 'con_tan_dir', 'con_tan_relV', 'edge_type'])


22

In [ ]:
graph.edges.keys()

dict_keys(['body_curr_disp', 'body_curr_dist', 'body_rest_disp', 'body_rest_dist', 'cable_disp', 'cable_dist', 'cable_dir', 'cable_relV', 'cable_relV_norm', 'cable_stiffness', 'cable_damping', 'cable_stiffness_mag', 'cable_damping_mag', 'cable_rest_len', 'con_dist', 'con_normal_dir', 'con_normal_relV', 'con_tan_dir', 'con_tan_relV', 'edge_type'])

In [ ]:
[v.shape for v in graph.edges.values()]

[(36, 3),
 (36, 1),
 (36, 3),
 (36, 1),
 (18, 3),
 (18, 1),
 (18, 3),
 (18, 3),
 (18, 1),
 (18, 1),
 (18, 1),
 (18, 1),
 (18, 1),
 (18, 1),
 (6, 1),
 (6, 3),
 (6, 1),
 (6, 3),
 (6, 1),
 (60,)]

### Method 1:
Let's try keeping everything in 1 graph -> may need to use jax.lax.cond or .switch to conditionally apply different mlp/concats based on edge type

First let's reorganize graph.edges hierarchy to have body, cable, con keys which each correspond to dicts holding
That edge types' feats

In [ ]:
def edge_reorg(edges):
    new_edges = {
        "body": {},
        "cable": {},
        "con": {},
    }

    for k, v in edges.items():
        if k.startswith("body_"):
            new_edges["body"][k] = v
        elif k.startswith("cable_"):
            new_edges["cable"][k] = v
        elif k.startswith("con_"):
            new_edges["con"][k] = v
        elif k == "edge_type":
            new_edges["edge_type"] = v
        else:
            raise ValueError(f"Unknown edge key: {k}")

    return new_edges

edges = edge_reorg(graph.edges)
edges['body'].keys()

dict_keys(['body_curr_disp', 'body_curr_dist', 'body_rest_disp', 'body_rest_dist'])

In [ ]:
graph = graph._replace(edges=edge_reorg(graph.edges))

In [ ]:
graph.edges.keys()

dict_keys(['body', 'cable', 'con', 'edge_type'])

In [ ]:
# For now doing everything by hand:
node_sz = sum(n.shape[-1] for n in graph.nodes.values())
body_sz = sum(n.shape[-1] for n in graph.edges['body'].values())
cable_sz = sum(n.shape[-1] for n in graph.edges['cable'].values())
con_sz = sum(n.shape[-1] for n in graph.edges['con'].values())

latent_dim = 128

# Make my mlp's
keys = jax.random.split(jax.random.PRNGKey(0), 4)
node_enc_mlp=eqx.nn.MLP(node_sz, latent_dim, 256, 2, key=keys[0])
body_enc_mlp=jraph.concatenated_args(eqx.nn.MLP(body_sz, latent_dim, 256, 2, key=keys[1]))
cable_enc_mlp=jraph.concatenated_args(eqx.nn.MLP(cable_sz, latent_dim, 256, 2, key=keys[2]))
con_enc_mlp=jraph.concatenated_args(eqx.nn.MLP(con_sz, latent_dim, 256, 2, key=keys[3]))


@jraph.concatenated_args
def encode_nodes(feats):
  # Need jax.vmap b/c node_mlp just expects input of shape (node_sz,), not (batch_sz, node_sz)
  return jax.vmap(node_enc_mlp)(feats)

def encode_edges(edges):
  edges =  {
      "body": jax.vmap(body_enc_mlp)(edges['body']),
      "cable": jax.vmap(cable_enc_mlp)(edges['cable']),
      "con": jax.vmap(con_enc_mlp)(edges['con']),
      "edge_type": edges['edge_type']
  }
  return edges

encoded_edges = encode_edges(graph.edges)


In [ ]:
# Now we want to concat this into 1 big edge matrix
np.concat([encoded_edges['body'], encoded_edges['cable'], encoded_edges['con']], axis=0).shape

(60, 128)

In [ ]:
# So now we can redefine our encode_edges fn:
def encode_edges(edges):
  edges =  {
      "body": jax.vmap(body_enc_mlp)(edges['body']),
      "cable": jax.vmap(cable_enc_mlp)(edges['cable']),
      "con": jax.vmap(con_enc_mlp)(edges['con']),
      "edge_type": edges['edge_type']
  }
  edge_matrix = np.concat([edges['body'], edges['cable'], edges['con']], axis=0)
  # concat edge types to last col of edge matrix
  edge_matrix = np.concatenate([edge_matrix, np.reshape(edges['edge_type'], shape=(edges['edge_type'].shape[0], 1))], axis=-1)
  return edge_matrix

encoded_edges = encode_edges(graph.edges)


In [ ]:
encoded_edges.shape

(60, 129)

In [ ]:
graph_encoder = jraph.GraphMapFeatures(
    embed_edge_fn = encode_edges,
    embed_node_fn = encode_nodes,
)

enc_graph = graph_encoder(graph)

In [ ]:
enc_graph.nodes.shape, enc_graph.edges.shape, enc_graph.edges[:, :latent_dim].shape

((22, 128), (60, 129), (60, 128))

Seems like what we did was both successful and pretty efficient, lets stick with this for now.

### Message Passing

Here's what we want to do for the processing step:
1. For every edge:
   - Concat edge + it's nodes and pass this through a NN to get new edge feats

2. For every Node:
   - Get all of it's edges
   - Aggregate all of them
   - concat and pass through NN

In [ ]:
# Let's start by testing out what interaction network gives us

edge_update_mlp = eqx.nn.MLP(latent_dim*3, latent_dim, 256, 2, key=keys[0])

def update_edge_fn(
      edge_features,
      sender_node_features,
      receiver_node_features,
      globals_):
  del globals_ # don't use these
  inp = np.concatenate([edge_features[:, :latent_dim], sender_node_features, receiver_node_features], axis=-1)
  return jax.vmap(edge_update_mlp)(inp)


node_update_mlp = eqx.nn.MLP(latent_dim*3, latent_dim, 256, 2, key=keys[1])

def update_node_fn(
      node_features,
      aggregated_sender_edge_features,
      aggregated_receiver_edge_features,
      globals_):
  del globals_ # don't use these
  print(f'Aggregated sender edges shape {aggregated_sender_edge_features.shape}')
  print(f'Aggregated reciever edges shape {aggregated_receiver_edge_features.shape}')
  inp = np.concatenate([node_features, aggregated_sender_edge_features, aggregated_receiver_edge_features], axis=-1)
  return jax.vmap(node_update_mlp)(inp)


aggregate_edges_for_nodes_fn = jraph.segment_sum


processor = jraph.GraphNetwork(
    update_edge_fn = update_edge_fn,
    update_node_fn = update_node_fn,
    aggregate_edges_for_nodes_fn = aggregate_edges_for_nodes_fn,
)

processed = processor(enc_graph)

Aggregated sender edges shape (22, 128)
Aggregated reciever edges shape (22, 128)


In [ ]:
processed.nodes.shape, processed.edges.shape

((22, 128), (60, 128))

In [ ]:
# Ok cool so we figured out how to use the basic GraphNetwork, now we need to use a more advanced aggregation function
# to accommodate the different types of edges


# First change the update edge fn to return edges of shape (n_edge, latent dim + 1)
def update_edge_fn(
      edge_features,
      sender_node_features,
      receiver_node_features,
      globals_):
  inp = np.concatenate([edge_features[:, :latent_dim], sender_node_features, receiver_node_features], axis=-1)
  edge_features.at[:, :latent_dim].set(jax.vmap(edge_update_mlp)(inp))
  return edge_features


# For now don't update nodes yet
def update_node_fn(
      node_features,
      aggregated_sender_edge_features,
      aggregated_receiver_edge_features,
      globals_):
  del globals_ # don't use these
  print(f'Aggregated sender edges shape {aggregated_sender_edge_features.shape}')
  print(f'Aggregated reciever edges shape {aggregated_receiver_edge_features.shape}')
  return node_features

# Hopefully now edge data that's passed is a dict
def aggregate_edges_fn(
    edge_data,
    node_idx,
    n_nodes):
  # Use a specialized version of segment_sum in order to aggregate different edge types
  edge_types = edge_data[:, -1]
  feats = edge_data[:, :latent_dim]

  segment_ids = node_idx * 3 + edge_types
  return jraph.segment_sum(feats, segment_ids.astype(int), n_nodes*3)


processor = jraph.GraphNetwork(
    update_edge_fn = update_edge_fn,
    update_node_fn = update_node_fn,
    aggregate_edges_for_nodes_fn = aggregate_edges_fn,
)

processed = processor(enc_graph)

Aggregated sender edges shape (66, 128)
Aggregated reciever edges shape (66, 128)


In [ ]:
# Now that we are aggregating edges correctly time to figure out the node update:

# We have node feats of shape (n_nodes, latent dim), we have aggregated edges of shape (n_nodes, 3, latent dim)
# We want to for each node -> concat it and it's aggregated nodes -> flatten this and send through node update mlp

node_update_mlp = eqx.nn.MLP(latent_dim*4, latent_dim, 256, 2, key=keys[1])

# First change the update edge fn to return edges of shape (n_edge, latent dim + 1)
def update_edge_fn(
      edge_features,
      sender_node_features,
      receiver_node_features):
  inp = np.concatenate([edge_features[:, :latent_dim], sender_node_features, receiver_node_features], axis=-1)
  edge_features.at[:, :latent_dim].set(jax.vmap(edge_update_mlp)(inp))
  return edge_features


def update_node_fn(
      node_features,
      agg_reciever_f):

  agg_edges = np.reshape(agg_reciever_f, shape=(node_features.shape[0],3, -1))
  node_feats = np.reshape(node_features, shape=(node_features.shape[0], 1, -1))
  inp = np.concatenate([node_feats, agg_edges], axis=1)
  inp = np.reshape(inp, shape=(node_feats.shape[0], -1))
  return jax.vmap(node_update_mlp)(inp)



processor = jraph.InteractionNetwork(
    update_edge_fn = update_edge_fn,
    update_node_fn = update_node_fn,
    aggregate_edges_for_nodes_fn = aggregate_edges_fn,
    include_sent_messages_in_node_update = False,
)

processed = processor(enc_graph)

In [ ]:
processed.nodes.shape, processed.edges.shape # looks like this worked!!!

((22, 128), (60, 129))

In [ ]:
# Lastly we just need to decode:
node_dec_mlp = eqx.nn.MLP(latent_dim, 3, 256, 2, key=keys[2])

decoder = jraph.GraphMapFeatures(
    embed_node_fn = jax.vmap(node_dec_mlp),
)

decoded = decoder(processed)

In [ ]:
decoded.nodes.shape, decoded.edges.shape # yay we did it!!!

((22, 3), (60, 129))

### Packaging this into a Module

Now that we've written out the whole forward pipeline let's build this into an equinox Module.

The whole while I will be trying to jit compile as much as I can for performance and then will try to write a training loop with this


Initialization:
Here's what I want to create at init:
1. Encoder MLPs for the node + diff edge types
2. Edge update and Node update MLPs
3. Node decoder MLP


Graph shape:
1. Nodes can stay the same
2. For the edges, should be updated to be a dict with keys body, cable, and con corresponding to dict values as well as an edge type key holding the type of each edge


In [ ]:
# Defining the Encoder class:
class Encoder(eqx.Module):
  node_enc_mlp: eqx.nn.MLP
  body_enc_mlp: eqx.nn.MLP
  cable_enc_mlp: eqx.nn.MLP
  con_enc_mlp: eqx.nn.MLP
  _jraph_encoder: jraph.GraphMapFeatures # Store the actual jraph object

  def __call__(self, graph):
    """
    Assumes graph input adheres to the node sz, body sz, etc. inputted when you call create_encoder
    """
    return self._jraph_encoder(graph)

In [ ]:
def create_encoder(
    latent_dim: int,
    hidden_sz: int,
    mlp_depth: int,
    node_sz: int,
    body_sz: int,
    cable_sz: int,
    con_sz: int,
    key: jax.random.PRNGKey) -> Encoder:
  """
  To create our encoder we first create 4 MLPs:
  One for nodes and each edge type
   - These MLPs will concatenate the array args they recieve by last axis
   - They will encode both nodes as well as edges into the latent dim space
   - For edges we concat onto the last column edge types as jraph doesn't support passing dicts to its aggregate fn
   - We package the encoder and all it's MLPs into a Encoder object

  The output after encoding (assuming that you give a correct graph input) is that
  edges will be of shape (n_edge, latent_dim+1) and nodes will be of shape (n_node, latent_dim)
  """
  keys = jax.random.split(key, 4)
  node_enc_mlp=eqx.nn.MLP(node_sz, latent_dim, hidden_sz, mlp_depth, key=keys[0])
  body_enc_mlp=eqx.nn.MLP(body_sz, latent_dim, hidden_sz, mlp_depth, key=keys[1])
  cable_enc_mlp=eqx.nn.MLP(cable_sz, latent_dim, hidden_sz, mlp_depth, key=keys[2])
  con_enc_mlp=eqx.nn.MLP(con_sz, latent_dim, hidden_sz, mlp_depth, key=keys[3])

  @jraph.concatenated_args
  def encode_nodes(feats):
    # Need jax.vmap b/c node_mlp just expects input of shape (node_sz,), not (batch_sz, node_sz)
    return jax.vmap(node_enc_mlp)(feats)

  def encode_edges(edges):
    edges =  {
        "body": jax.vmap(jraph.concatenated_args(body_enc_mlp))(edges['body']),
        "cable": jax.vmap(jraph.concatenated_args(cable_enc_mlp))(edges['cable']),
        "con": jax.vmap(jraph.concatenated_args(con_enc_mlp))(edges['con']),
        "edge_type": edges['edge_type']
    }
    edge_matrix = np.concat([edges['body'], edges['cable'], edges['con']], axis=0)
    # concat edge types to last col of edge matrix
    edge_matrix = np.concatenate([edge_matrix, np.reshape(edges['edge_type'], shape=(edges['edge_type'].shape[0], 1))], axis=-1)
    return edge_matrix

  jraph_enc =  jraph.GraphMapFeatures(
      embed_edge_fn = encode_edges,
      embed_node_fn = encode_nodes,
  )
  return Encoder(node_enc_mlp, body_enc_mlp, cable_enc_mlp, con_enc_mlp, jraph_enc)



In [ ]:
enc = create_encoder(latent_dim=128, hidden_sz=128, mlp_depth=1,
              node_sz=node_sz, body_sz=body_sz, cable_sz=cable_sz,
              con_sz=con_sz, key=jax.random.key(0))

enc(graph).nodes.shape, enc(graph).edges.shape

((22, 128), (60, 129))

In [ ]:
# Next we define the Processor class
class MsgPass(eqx.Module):
  edge_update_mlp: eqx.nn.MLP
  node_update_mlp: eqx.nn.MLP
  _jraph_processor: jraph.InteractionNetwork

  def __call__(self, graph):
    return self._jraph_processor(graph)


In [ ]:
def make_graph_net(
    latent_dim: int,
    hidden_sz: int,
    mlp_depth: int,
    key: jax.random.PRNGKey) -> jraph.GraphNetwork:
  """
  How our graph network works:
  1. We will create an edge update MLP which takes as input a concatenation of edge + sender and reciever node feats
    - We pass the first latent dim cols of edge feats to this MLP, keeping the edge types array the same
  2. We aggregate edges based on edge type
     - achieve this by using a specialized version of segment_sum (more comments and explanation below)
  3. We create a node update MLP which takes nodes feats + aggregated edge feats of all edge types
   - So the final input vector has latent dim * 4 features

  All of this is packaged into a jraph.GraphNetwork object
  """
  keys = jax.random.split(key, 2)

  edge_update_mlp = eqx.nn.MLP(latent_dim*3, latent_dim, hidden_sz, mlp_depth, key=keys[0])
  node_update_mlp = eqx.nn.MLP(latent_dim*4, latent_dim, hidden_sz, mlp_depth, key=keys[1])

  def update_edge_fn(
      edge_features,
      sender_node_features,
      receiver_node_features):
    inp = np.concatenate([edge_features[:, :latent_dim], sender_node_features, receiver_node_features], axis=-1)
    edge_features.at[:, :latent_dim].set(jax.vmap(edge_update_mlp)(inp))
    return edge_features

  def aggregate_edges_fn(
    edge_data,
    node_idx,
    n_nodes):
    # Use a specialized version of segment_sum in order to aggregate different edge types
    edge_types = edge_data[:, -1]
    feats = edge_data[:, :latent_dim]

    # Instead of just using n_nodes as number of segments we want
    # we say that for each node there's 3 segments we want, i.e. 1 for each edge type
    num_segments = n_nodes * 3

    # The segment ids are then the node idx scaled by 3 + the edge type
    segment_ids = node_idx * 3 + edge_types # (0, 5) 6
    return jraph.segment_sum(feats, segment_ids.astype(int), num_segments) # gives shape (num_segments, latent dim)

  def update_node_fn(
      node_features,
      agg_reciever_feats):
    # For now we consider aggregated edges the sum of both sender and reciever edges for each node
    agg_edges = np.reshape(agg_reciever_feats, shape=(node_features.shape[0],3, -1))

    # Add an extra dimension into node features so that we can concat them with their aggregated edge feats
    node_feats = np.reshape(node_features, shape=(node_features.shape[0], 1, -1))
    inp = np.concatenate([node_feats, agg_edges], axis=1)

    # Lastly we flatten the input to get shape (n_nodes, concatted node and agg edge feats) and send this into the MLP
    inp = np.reshape(inp, shape=(node_feats.shape[0], -1))
    return jax.vmap(node_update_mlp)(inp)

  net = jraph.InteractionNetwork(
      update_edge_fn = update_edge_fn,
      update_node_fn = update_node_fn,
      aggregate_edges_for_nodes_fn = aggregate_edges_fn,
      include_sent_messages_in_node_update=False
  )

  return MsgPass(edge_update_mlp, node_update_mlp, net)



In [ ]:
msg_pass = make_graph_net(latent_dim=128, hidden_sz=128, mlp_depth=1, key=jax.random.key(0))
msg_pass(enc(graph)).nodes.shape, msg_pass(enc(graph)).edges.shape

((22, 128), (60, 129))

In [ ]:
class CustomGNN(eqx.Module):
  encoder: Encoder
  processor: list[MsgPass]
  decoder_mlp: eqx.nn.MLP
  decoder: jraph.GraphMapFeatures

  def __init__(self,
               latent_dim,
               hidden_dim,
               mlp_depth,
               n_mp_steps,
               node_sz, body_sz, cable_sz, con_sz, key=jax.random.key(0)):
    keys = jax.random.split(key, 3)
    self.encoder = create_encoder(latent_dim, hidden_dim, mlp_depth,
                                  node_sz, body_sz, cable_sz, con_sz, keys[0])

    self.processor = []
    for i in range(n_mp_steps):
      self.processor.append(make_graph_net(latent_dim, hidden_dim, mlp_depth, keys[1]))

    self.decoder_mlp = eqx.nn.MLP(latent_dim, 3, hidden_dim, mlp_depth, key=keys[2])
    self.decoder = jraph.GraphMapFeatures(
        embed_node_fn = jax.vmap(self.decoder_mlp),
    )



  def __call__(self, graph: jraph.GraphsTuple) -> jraph.GraphsTuple:
    latent_graph = self.encoder(graph)

    inp_graph = latent_graph

    for mp_step in self.processor:

      proc_graph = mp_step(inp_graph)

      # Add residual connections
      proc_graph._replace(nodes = proc_graph.nodes + inp_graph.nodes)
      proc_graph._replace(edges = proc_graph.edges + inp_graph.edges)

      inp_graph = proc_graph

    node_dv_preds = self.decoder(proc_graph).nodes
    return node_dv_preds





In [ ]:
node_sz = sum(n.shape[-1] for n in graph.nodes.values())
body_sz = sum(n.shape[-1] for n in graph.edges['body'].values())
cable_sz = sum(n.shape[-1] for n in graph.edges['cable'].values())
con_sz = sum(n.shape[-1] for n in graph.edges['con'].values())

GNN = CustomGNN(latent_dim=128, n_mp_steps=2, hidden_dim=128,
                mlp_depth=2, node_sz=node_sz, body_sz=body_sz, cable_sz=cable_sz, con_sz=con_sz)

In [ ]:
dvs = GNN(graph)

In [ ]:
dvs.shape

(22, 3)

In [ ]:
import optax

def MSE_loss(model, graph, targets):
  preds = model(graph)
  return optax.l2_loss(preds, targets).mean()

MSE_loss(GNN, graph, np.zeros(shape=(22, 3)))

Array(73.88519, dtype=float32)

In [ ]:
from typing import Callable, List, Tuple
import optax
from tqdm.auto import tqdm

def fit(
    model: eqx.Module,
    data: np.array,
    loss_fn: Callable,
    optimizer: optax.GradientTransformation,
    steps: int,
    progress_bar: bool = True,
  ) -> Tuple[List[eqx.Module], List]:
  """
  Fit model to data

  data should have shape (batch, :), at least a 2D tensor

  returns updated model + loss training history
  """
  # Set up the state of the optimizer, filtering for weights which are inexact (which is apparently anything that is floating point)
  opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

  # Here we call filter_value_and_grad s.t. it returns pair (value, grad). We jit compile it too to be faster
  dloss = eqx.filter_jit(eqx.filter_value_and_grad(loss_fn))

  @eqx.filter_jit
  def step(model, opt_state, data):
    """
    One step of training loop, jitted to run faster
    """

    loss, grads = dloss(model, graph, data)

    # Next we compute the updates and obtain optimizer state
    updates, opt_state = optimizer.update(grads, opt_state)

    # Apply updates to the model
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

  loss_hist = []
  iterator = range(steps)
  if progress_bar: iterator = tqdm(iterator)

  for _ in iterator:
    model, opt_state, loss = step(model, opt_state, data)
    loss_hist.append(loss)
  return model, loss_hist

In [ ]:
normal_data = jax.random.normal(jax.random.key(28), shape=(22, 3))

fit(GNN, normal_data, MSE_loss, optax.adam(1e-3), 1)

  0%|          | 0/1 [00:00<?, ?it/s]

(CustomGNN(
   encoder=Encoder(
     node_enc_mlp=MLP(
       layers=(
         Linear(
           weight=f32[128,22],
           bias=f32[128],
           in_features=22,
           out_features=128,
           use_bias=True
         ),
         Linear(
           weight=f32[128,128],
           bias=f32[128],
           in_features=128,
           out_features=128,
           use_bias=True
         ),
         Linear(
           weight=f32[128,128],
           bias=f32[128],
           in_features=128,
           out_features=128,
           use_bias=True
         )
       ),
       activation=<PjitFunction of <function relu at 0x785151348720>>,
       final_activation=<function <lambda>>,
       use_bias=True,
       use_final_bias=True,
       in_size=22,
       out_size=128,
       width_size=128,
       depth=2
     ),
     body_enc_mlp=MLP(
       layers=(
         Linear(
           weight=f32[128,8],
           bias=f32[128],
           in_features=8,
           out_features=1

In [ ]:
dloss = eqx.filter_jit(eqx.filter_value_and_grad(MSE_loss))

loss, grads = dloss(GNN, graph, normal_data)

optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(GNN, eqx.is_inexact_array))

In [ ]:
updates, opt_state = optimizer.update(grads, opt_state)

In [ ]:
jax.tree_util.tree_structure(grads), jax.tree_util.tree_leaves(grads)

(PyTreeDef(CustomNode(CustomGNN[('encoder', 'processor', 'decoder_mlp', 'decoder'), ()], [CustomNode(Encoder[('node_enc_mlp', 'body_enc_mlp', 'cable_enc_mlp', 'con_enc_mlp', '_jraph_encoder'), ()], [CustomNode(MLP[('layers', 'activation', 'final_activation'), (('use_bias', True), ('use_final_bias', True), ('in_size', 22), ('out_size', 128), ('width_size', 128), ('depth', 2))], [(CustomNode(Linear[('weight', 'bias'), (('in_features', 22), ('out_features', 128), ('use_bias', True))], [*, *]), CustomNode(Linear[('weight', 'bias'), (('in_features', 128), ('out_features', 128), ('use_bias', True))], [*, *]), CustomNode(Linear[('weight', 'bias'), (('in_features', 128), ('out_features', 128), ('use_bias', True))], [*, *])), None, None]), CustomNode(MLP[('layers', 'activation', 'final_activation'), (('use_bias', True), ('use_final_bias', True), ('in_size', 8), ('out_size', 128), ('width_size', 128), ('depth', 2))], [(CustomNode(Linear[('weight', 'bias'), (('in_features', 8), ('out_features', 1

Ok so possibly the reason why this error is occuring is b/c when we call grad on our loss function it computes gradients of the function w.r.t. to the first argument (which is preds). Try instead to have model, data, targ structure

 - While this works to track the proper gradients, using jax.filter_grad() on our loss function didn't work
 - This is because under the hood eqx uses custom derivatives so need to use eqx.filter_grad()

In [ ]:
def MSE_loss(model, graph, data):
  preds = model(graph)
  return optax.l2_loss(preds, data).mean()

dloss = eqx.filter_grad(MSE_loss)

dloss(GNN, graph, normal_data)

CustomGNN(
  encoder=Encoder(
    node_enc_mlp=MLP(
      layers=(
        Linear(
          weight=f32[128,22],
          bias=f32[128],
          in_features=22,
          out_features=128,
          use_bias=True
        ),
        Linear(
          weight=f32[128,128],
          bias=f32[128],
          in_features=128,
          out_features=128,
          use_bias=True
        ),
        Linear(
          weight=f32[128,128],
          bias=f32[128],
          in_features=128,
          out_features=128,
          use_bias=True
        )
      ),
      activation=None,
      final_activation=None,
      use_bias=True,
      use_final_bias=True,
      in_size=22,
      out_size=128,
      width_size=128,
      depth=2
    ),
    body_enc_mlp=MLP(
      layers=(
        Linear(
          weight=f32[128,8],
          bias=f32[128],
          in_features=8,
          out_features=128,
          use_bias=True
        ),
        Linear(
          weight=f32[128,128],
          bias=f32

### To Do:
1. Refactor to be able to track param grads
2. Use interaction network and set sender_msgs = False
3. Ensure that you're using residual connections b/w graphs
4. Add normalizer for normalizing input graph (before encoder)
5. Add a layer norm at the end of every mlp except decoder

Data:
 - GT rod states -> map to node positions
 - to get GT delta vel
  - curr rod states -> get_prev()
  - Map curr rod pos/quat (and prev) to node pos
  - v = curr - prev / dt
  - Assume robot is initially at rest (delta v = 0 at start)
  - Take all the v's (v=0, v=1, v=2) -> (dv=0, dv = 1, dv=1)

For me:
 - From the data I also get motor speeds + control inputs
 - call applyControl(...) -> new rest lengths
 - call updateState on robot
 - check new rest lengths = data.curr_rest_lengths[i+1]
 - From this I get N graph inputs + N dvs
  - Accum mu and std from r.b. states
  - shuffle into random batches and train


In [ ]:
# Want to work on the accumulated normalizer for input graphs and their features

# Let's see how we can get just the number of features -> notice that the graph is really just a pytree
type(graph.nodes)

dict

In [ ]:
for arr in eqx.filter(jax.tree.leaves(graph.nodes), eqx.is_inexact_array):
  print(arr.shape[-1])

3
3
1
1
3
1
1
3
3
3


In [ ]:
eqx.filter(jax.tree.leaves(graph.edges), eqx.is_inexact_array)


[array([[-0.43943644, -0.05933844, -0.34609365],
        [ 0.43943644,  0.05933844,  0.34609365],
        [ 0.83004665,  0.11208381,  0.6537327 ],
        [-0.83004665, -0.11208381, -0.6537327 ],
        [ 0.83004665,  0.11208378,  0.65373266],
        [-0.83004665, -0.11208378, -0.65373266],
        [ 0.43943635,  0.05933848,  0.3460939 ],
        [-0.43943635, -0.05933848, -0.3460939 ],
        [-0.08788729, -0.01186765, -0.06921864],
        [ 0.08788729,  0.01186765,  0.06921864],
        [ 0.08788732,  0.01186767,  0.06921864],
        [-0.08788732, -0.01186767, -0.06921864],
        [-0.08151895, -0.54360783,  0.11937928],
        [ 0.08151895,  0.54360783, -0.11937928],
        [ 0.15398018,  1.0268146 , -0.2254945 ],
        [-0.15398018, -1.0268146 ,  0.2254945 ],
        [ 0.15398023,  1.0268148 , -0.22549456],
        [-0.15398023, -1.0268148 ,  0.22549456],
        [ 0.08151895,  0.54360783, -0.11937949],
        [-0.08151895, -0.54360783,  0.11937949],
        [-0.01630381

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2], dtype=int32)